# November Final Protected-Test Evaluation

This notebook performs the one-time final evaluation of the October-approved top-75 CatBoost candidate on the protected test partition.

## Evaluation contract

- The source, split, feature list, and serialized model are frozen before execution.
- This notebook does not retrain, tune, select features, or compare new configurations.
- Baselines are learned from training `loss` only and scored on the same test claims.
- MAE is reported in the original `loss` units.
- After this score, the model must not be changed and rescored without a documented decision.

The workflow is intentionally separated by function: locate artifacts, verify frozen inputs, establish partitions, prepare the model, score once, inspect high-loss behavior, and save the record.


## 1. Locate the project and define paths

The notebook finds the repository by walking upward from the current working directory. Input artifacts are discovered by filename and checked for uniqueness, so the notebook does not depend on a personal absolute path.


In [7]:
from importlib.metadata import version
from pathlib import Path
import hashlib
import json
import platform

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

SOURCE_SHA256 = "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03"
MANIFEST_SHA256 = "de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f37b87a6235f582a64b"
EXPECTED_COUNTS = {"train": 131_822, "validation": 28_248, "test": 28_248}
CAT_COLS = [f"cat{i}" for i in range(1, 117)]
TARGET_COLUMN = "loss"

repo_root = Path.cwd()
while not (repo_root / "data" / "allstate_claims_data.csv").exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not locate repository root")
    repo_root = repo_root.parent


def find_unique(pattern):
    matches = sorted(repo_root.glob(pattern))
    assert len(matches) == 1, f"Expected one match for {pattern!r}; found {matches}"
    return matches[0]


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


source_path = find_unique("data/allstate_claims_data.csv")
manifest_path = find_unique("notebooks/**/partition_manifest.csv")
model_path = find_unique("notebooks/**/final_top75_catboost_model.cbm")
feature_path = find_unique("notebooks/**/final_catboost_features.csv")
output_dir = repo_root / "notebooks" / "final-deliverables" / "November" / "Gate 2" / "test_evaluation_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

print("Repository root detected from the current workspace.")
print("Source:", source_path.relative_to(repo_root))
print("Manifest:", manifest_path.relative_to(repo_root))
print("Model:", model_path.relative_to(repo_root))


Repository: /Users/Jonathan/Documents/GitHub/Allstate-1A-predicting-auto-claims-severity
Source: data/allstate_claims_data.csv
Manifest: notebooks/final-deliverables/October/Gate 1/partition_manifest.csv
Model: notebooks/liam-stapley/model_experiment_outputs/final_top75_catboost_model.cbm


## 2. Verify frozen inputs

The source and manifest hashes identify the exact modeling data and claim-level split approved for October. The notebook stops before scoring if either artifact changes.


In [8]:
assert sha256_file(source_path) == SOURCE_SHA256
assert sha256_file(manifest_path) == MANIFEST_SHA256
assert model_path.exists() and feature_path.exists()
print("Frozen source, manifest, model, and feature list verified.")


Frozen source, manifest, model, and feature list verified.


## 3. Establish the frozen partition boundary

The manifest is joined one-to-one with the source. Counts and ID disjointness are checked before the test rows are exposed for scoring. The test target is not used to fit preprocessing or baselines.


In [9]:
data = pd.read_csv(source_path)
manifest = pd.read_csv(manifest_path)
assert data["id"].is_unique and manifest["id"].is_unique
assert len(data) == sum(EXPECTED_COUNTS.values())

df = data.merge(manifest, on="id", how="inner", validate="one_to_one")
partition_counts = df["partition"].value_counts().to_dict()
assert partition_counts == EXPECTED_COUNTS, partition_counts

partitions = {
    name: df[df["partition"] == name].copy().reset_index(drop=True)
    for name in EXPECTED_COUNTS
}
train, validation, test = (partitions[name] for name in ("train", "validation", "test"))

for left, right in ((train, validation), (train, test), (validation, test)):
    assert set(left["id"]).isdisjoint(right["id"])

print("Partition counts:", partition_counts)
print("Train/validation/test ID overlap: none")
print("Protected test claims authorized for final scoring:", len(test))


Partition counts: {'train': 131822, 'test': 28248, 'validation': 28248}
Train/validation/test ID overlap: none
Protected test claims authorized for final scoring: 28248


## 4. Load the approved model and prepare features

The final artifact is loaded directly. CatBoost receives the same 75 predictors and categorical-string representation used during training. No `fit`, cross-validation, feature ranking, or tuning occurs here.


In [10]:
selected_features = pd.read_csv(feature_path)["feature"].tolist()
assert len(selected_features) == 75 and len(set(selected_features)) == 75
assert not [feature for feature in selected_features if feature not in test.columns]
selected_cat_cols = [feature for feature in selected_features if feature in CAT_COLS]


def prepare_catboost_features(frame):
    X = frame[selected_features].copy()
    X[selected_cat_cols] = X[selected_cat_cols].astype(str)
    return X


model = CatBoostRegressor()
model.load_model(str(model_path))
params = model.get_all_params()
assert model.tree_count_ == 500
assert int(params["depth"]) == 8
assert np.isclose(float(params["learning_rate"]), 0.08)
assert str(params["loss_function"]).startswith("MAE")
assert int(params["random_seed"]) == 42

X_test = prepare_catboost_features(test)
print("Approved CatBoost configuration verified.")
print(f"Features: {len(selected_features)}; categorical features: {len(selected_cat_cols)}")


Approved CatBoost configuration verified.
Features: 75; categorical features: 61


## 5. Score the protected test partition once

This is the final held-out score. Training mean and median constants are computed from training `loss` only, then compared with CatBoost on exactly the same test claims.


In [11]:
y_test = test[TARGET_COLUMN].to_numpy()
training_mean = float(train[TARGET_COLUMN].mean())
training_median = float(train[TARGET_COLUMN].median())
mean_baseline_mae = float(mean_absolute_error(y_test, np.full(len(test), training_mean)))
median_baseline_mae = float(mean_absolute_error(y_test, np.full(len(test), training_median)))

test_predictions = model.predict(X_test)
assert len(test_predictions) == EXPECTED_COUNTS["test"]
assert np.isfinite(test_predictions).all()
test_mae = float(mean_absolute_error(y_test, test_predictions))

comparison = pd.DataFrame({
    "model": ["training_mean_constant", "training_median_constant", "final_top75_catboost"],
    "test_mae": [mean_baseline_mae, median_baseline_mae, test_mae],
})
comparison["improvement_vs_mean_pct"] = (mean_baseline_mae - comparison["test_mae"]) / mean_baseline_mae * 100
comparison["improvement_vs_median_pct"] = (median_baseline_mae - comparison["test_mae"]) / median_baseline_mae * 100

results = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "loss": y_test,
    "prediction": test_predictions,
})
results["absolute_error"] = (results["loss"] - results["prediction"]).abs()

print(comparison.to_string(index=False))
print("Finite predictions:", bool(np.isfinite(test_predictions).all()))
print("Negative predictions:", int((test_predictions < 0).sum()))


                   model    test_mae  improvement_vs_mean_pct  improvement_vs_median_pct
  training_mean_constant 1963.447313                 0.000000                  -8.697234
training_median_constant 1806.345236                 8.001339                   0.000000
    final_top75_catboost 1141.571999                41.858791                  36.802114
Finite predictions: True
Negative predictions: 0


## 6. Diagnose high-loss claims

The p90 and p99 thresholds are calculated from training `loss` only. Segment MAE is diagnostic context; overall MAE remains the primary metric. The p99 segment is small, so differences there should be interpreted cautiously.


In [12]:
p90_threshold = float(train[TARGET_COLUMN].quantile(0.90))
p99_threshold = float(train[TARGET_COLUMN].quantile(0.99))
results["loss_segment"] = pd.cut(
    results["loss"],
    bins=[-np.inf, p90_threshold, p99_threshold, np.inf],
    labels=["below_p90", "p90_to_p99", "p99_and_above"],
    right=False,
)
tail_diagnostics = (
    results.groupby("loss_segment", observed=False)
    .agg(
        claims=("id", "size"),
        mae=("absolute_error", "mean"),
        mean_loss=("loss", "mean"),
        mean_prediction=("prediction", "mean"),
    )
    .reset_index()
)

print(f"Training p90 threshold: {p90_threshold:.6f}")
print(f"Training p99 threshold: {p99_threshold:.6f}")
print(tail_diagnostics.to_string(index=False))


Training p90 threshold: 6401.738000
Training p99 threshold: 14024.143300
 loss_segment  claims         mae    mean_loss  mean_prediction
    below_p90   25424  870.383414  2298.869423      2311.250645
   p90_to_p99    2559 2982.064535  8715.065111      6082.917324
p99_and_above     265 9386.395303 18769.694868      9449.177376


## 7. Save the final evaluation record

These artifacts preserve the exact rows, predictions, comparisons, diagnostics, hashes, model configuration, and environment used for the November score.


In [13]:
prediction_path = output_dir / "final_test_predictions.csv"
comparison_path = output_dir / "final_test_model_comparison.csv"
tail_path = output_dir / "final_test_tail_diagnostics.csv"
receipt_path = output_dir / "final_test_evaluation_receipt.json"

results.to_csv(prediction_path, index=False)
comparison.to_csv(comparison_path, index=False)
tail_diagnostics.to_csv(tail_path, index=False)

receipt = {
    "evaluation_type": "one-time protected-test evaluation",
    "modeling_data_version": "OCT-MD-001",
    "split_version": "OCT-SPLIT-001",
    "source_sha256": SOURCE_SHA256,
    "manifest_sha256": MANIFEST_SHA256,
    "partition_counts": {key: int(value) for key, value in partition_counts.items()},
    "test_claims": int(len(test)),
    "test_mae": test_mae,
    "training_mean_constant_mae": mean_baseline_mae,
    "training_median_constant_mae": median_baseline_mae,
    "feature_count": int(len(selected_features)),
    "categorical_feature_count": int(len(selected_cat_cols)),
    "model_path": str(model_path.relative_to(repo_root)),
    "model_sha256": sha256_file(model_path),
    "feature_path": str(feature_path.relative_to(repo_root)),
    "model_configuration": {
        "loss_function": str(params["loss_function"]),
        "depth": int(params["depth"]),
        "learning_rate": float(params["learning_rate"]),
        "iterations": int(model.tree_count_),
        "random_seed": int(params["random_seed"]),
    },
    "p90_training_threshold": p90_threshold,
    "p99_training_threshold": p99_threshold,
    "finite_predictions": bool(np.isfinite(test_predictions).all()),
    "negative_predictions": int((test_predictions < 0).sum()),
    "test_partition_used_for": ["final scoring", "final diagnostics"],
    "test_partition_used_for_fitting_or_tuning": False,
    "post_test_model_change_allowed": False,
    "library_versions": {
        "python": platform.python_version(),
        "numpy": version("numpy"),
        "pandas": version("pandas"),
        "catboost": version("catboost"),
        "scikit-learn": version("scikit-learn"),
    },
}
with open(receipt_path, "w", encoding="utf-8") as file:
    json.dump(receipt, file, indent=2)

print("Saved final evaluation artifacts to:", output_dir.relative_to(repo_root))


Saved final evaluation artifacts to: /Users/Jonathan/Documents/GitHub/Allstate-1A-predicting-auto-claims-severity/notebooks/final-deliverables/November/Gate 2/test_evaluation_outputs


## 8. Recorded November results

The executed frozen evaluation produced:

- **Final CatBoost test MAE:** `1,141.5720`
- Training-mean baseline MAE: `1,963.4473`
- Training-median baseline MAE: `1,806.3452`
- Improvement versus mean baseline: `41.86%`
- Improvement versus median baseline: `36.80%`
- Test claims: `28,248`
- Finite predictions: `True`; negative predictions: `0`

High-loss diagnostics used training thresholds of `6,401.738` for p90 and `14,024.1433` for p99. Test-segment MAE was `870.3834` below p90, `2,982.0645` from p90 to p99, and `9,386.3953` at or above p99.

The test result is the final held-out score for this frozen split and model artifact. It is not evidence of future-period performance, production reserve accuracy, causal importance, or guaranteed first-filing performance. Independent teammate reproduction is still required before marking November Gate 2 complete.
